In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install pyqtgraph PyOpenGL PyQt5


In [ ]:
import cv2 as cv  
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm, colors
import matplotlib

import os
import sys
from collections import deque
import time


#real time graphs
import pyqtgraph as pg
import pyqtgraph.opengl as gl
from pyqtgraph.Qt import QtCore #TIMER


In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "tk")          # notebook: live 3D window
except NameError:
    matplotlib.use(os.environ.get("MPLBACKEND", "TkAgg"))     # plain script
import matplotlib.pyplot as plt
 
CAM, SEAM = 0, 98          # camera index, columns cropped off the left
SCALE = 4                  # camera window size multiplier
ROWS, STEP = 80, 4         # time rows kept, column step
INTERVAL = 0.1            # seconds per 3D map update (bigger = slower)
FIGSIZE = (12, 8)          # 3D window size
 
cap = cv.VideoCapture(CAM)
if not cap.isOpened():
    raise RuntimeError("Could not open camera - try another CAM index")
 
plt.ion()
fig = plt.figure("Thermal 3D Map", figsize=FIGSIZE)
ax = fig.add_subplot(111, projection="3d")
rows = row = None
last = 0.0
 
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        gray = cv.cvtColor(frame[:, SEAM:], cv.COLOR_BGR2GRAY)
        cv.imshow("TC001", cv.resize(cv.applyColorMap(gray, cv.COLORMAP_INFERNO), None, fx=SCALE, fy=SCALE))
 
        line = gray.max(axis=0)[::STEP].astype(np.float32)     # hottest pixel per column
        row = line if row is None else np.maximum(row, line)   # keep peaks between updates
        if rows is None:
            rows = deque([np.zeros_like(line)] * ROWS, maxlen=ROWS)
            X, Y = np.meshgrid(np.arange(0, gray.shape[1], STEP), np.arange(ROWS))
 
        if time.monotonic() - last >= INTERVAL:
            last = time.monotonic()
            rows.append(row)
            row = None
            ax.clear()
            ax.plot_surface(X, Y, np.array(rows), cmap="inferno", vmin=0, vmax=255,
                            linewidth=0, antialiased=False, rcount=ROWS, ccount=X.shape[1])
            ax.set(xlabel="Image X", ylabel="Time", zlabel="Signal", zlim=(0, 255))
            ax.view_init(45, -70)
            plt.pause(0.001)
            if not plt.fignum_exists(fig.number):
                break
 
        if cv.waitKey(1) & 0xFF in (ord("q"), 27):
            break
finally:
    cap.release()
    cv.destroyAllWindows()
    plt.close("all")

In [ ]:
#Bounding box Code here and color bar 